In [1]:
import mysql.connector
from mysql.connector import Error


from config import DB_CONFIG

In [2]:
class DBManager:
    @staticmethod
    def get_connection():
        try:
            connection = mysql.connector.connect(**DB_CONFIG)

            if connection.is_connected():
                return connection
            
        except Error as e:
            print(f"Data base connection error: {e}")

        return None
    
    @staticmethod
    def execute_query(
        query: str,
        values: tuple=(),
        fetch_one: bool = False,
        fetch_all:bool=False,
        commit: bool = False
    ):
        """ returns fetched data or affected row count"""

        connection = None
        cursor = None

        try:
            connection = DBManager.get_connection()

            if not connection:
                return None
            
            cursor = connection.cursor()

            if commit:
                connection.commit()

            if fetch_one:
                return cursor.fetchone()
            
            if fetch_all:
                return cursor.fetchall()
            
            return cursor.rowcount
        
        except Error as e:
            if connection:
                connection.rollback()

            print(f"database error:  {e}")

            return None
        finally:
            if cursor:
                cursor.close()
            if connection:
                connection.close()


                
    

    

In [3]:
class Validator:

    """ handles input validation"""

    @staticmethod
    def validate_amount(amount: float) -> bool:
        return amount > 0
    
    @staticmethod
    def validate_name(name: str) -> bool:
        return len(name.strip()) >=3

In [ ]:
class AccountService:
    """handles business logic"""

    @staticmethod
    def create_account():
        print("create account")

        name = input("enter the customer name: ").strip()

        if not Validator.validate_name(name):
            print("invalid customer name")
            return
        try:
            balance = float(input("enter initial balance: "))
        except ValueError:
            print("invalid balance")
            return
        
        query = """ insert into accounts
        (name,balance)
        values(%s,%s)"""

        rows = DBManager.execute_query(
            query=query,
            values=(name,balance)
            commit=True
        )